# Handwritten Digit Recognition using CNN with MNIST

This notebook walks through the complete deep learning pipeline for classifying handwritten digits (0–9) using a Convolutional Neural Network (CNN).

**Topics covered:**
1. Import libraries
2. Load and explore the MNIST dataset
3. Data preprocessing
4. Build and compile the CNN
5. Train the model
6. Plot learning curves
7. Evaluate on the test set
8. Confusion matrix and classification report
9. Visualize predictions
10. Save the model
11. Predict on custom images

## Step 1: Import Libraries

We import TensorFlow/Keras for deep learning, NumPy and Pandas for data handling, Matplotlib and Seaborn for visualization, Scikit-learn for metrics, and OpenCV for custom image preprocessing.

In [ ]:
# Deep learning framework
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist

# Data and math libraries
import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Evaluation metrics
from sklearn.metrics import confusion_matrix, classification_report

# Image processing for custom inference
import cv2

# Display plots inline in the notebook
%matplotlib inline

# Print TensorFlow version for reproducibility
print(f"TensorFlow version: {tf.__version__}")

## Step 2: Load the MNIST Dataset

MNIST contains 70,000 grayscale images of handwritten digits. Keras downloads the dataset automatically on first use.

In [ ]:
# Load training and test splits
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Display dataset shapes
print(f"Training data shape:   {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Testing data shape:    {X_test.shape}")
print(f"Testing labels shape:  {y_test.shape}")
print(f"Number of classes:     {len(np.unique(y_train))}")
print(f"Pixel value range:     [{X_train.min()}, {X_train.max()}]")

In [ ]:
# Visualize 10 sample training images with their labels
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f'Label: {y_train[i]}')
    ax.axis('off')
plt.suptitle('Sample MNIST Training Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 3: Data Preprocessing

- **Normalize** pixel values from [0, 255] to [0.0, 1.0] for stable gradient descent
- **Reshape** images from (28, 28) to (28, 28, 1) to add a channel dimension for Conv2D layers

In [ ]:
# Normalize pixel values to [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Reshape: (28, 28) -> (28, 28, 1)
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

print(f"Final training shape: {X_train.shape}")
print(f"Final testing shape:  {X_test.shape}")
print(f"Pixel range (train):  [{X_train.min():.2f}, {X_train.max():.2f}]")

## Step 4: Build the CNN Model

Our architecture uses two convolutional blocks followed by a fully connected classifier:

| Layer | Description |
|-------|-------------|
| Conv2D (32) | Detects low-level features (edges, strokes) |
| MaxPooling2D | Reduces spatial dimensions |
| Conv2D (64) | Detects higher-level patterns |
| MaxPooling2D | Further dimensionality reduction |
| Flatten | Converts 2D feature maps to 1D vector |
| Dense (128) | Learns class-specific representations |
| Dropout (0.5) | Prevents overfitting |
| Dense (10, softmax) | Outputs probability for each digit |

In [ ]:
# Build the Sequential CNN model
model = keras.Sequential([
    # Input: 28x28 grayscale images
    layers.Input(shape=(28, 28, 1)),

    # Convolutional Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Convolutional Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Classifier Head
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax'),
], name='mnist_cnn')

# Print layer-by-layer summary
model.summary()

## Step 5: Compile the Model

- **Optimizer:** Adam — adaptive learning rate, works well for CNNs
- **Loss:** Sparse Categorical Crossentropy — for integer labels (0–9)
- **Metric:** Accuracy

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)
print("Model compiled successfully.")

## Step 6: Train the Model

Training configuration:
- **Epochs:** 10
- **Batch size:** 32
- **Validation split:** 10% of training data

Callbacks included: EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, TensorBoard.

In [ ]:
from pathlib import Path

# Project paths (notebook is in notebooks/ subfolder)
PROJECT_ROOT = Path('..').resolve()
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Training callbacks for robust training
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(
        filepath=str(MODELS_DIR / 'cnn_model.keras'),
        monitor='val_accuracy', save_best_only=True
    ),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6),
    keras.callbacks.TensorBoard(log_dir=str(RESULTS_DIR / 'tensorboard_logs')),
]

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

## Step 7: Plot Learning Curves

Visualize how accuracy and loss change across training epochs for both training and validation sets.

In [ ]:
# Accuracy plot
plt.figure(figsize=(10, 6))
plt.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
plt.title('Accuracy vs. Epochs', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'accuracy_plot.png', dpi=150)
plt.show()

# Loss plot
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
plt.title('Loss vs. Epochs', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'loss_plot.png', dpi=150)
plt.show()

## Step 8: Evaluate on the Test Set

Measure final performance using test loss, accuracy, and a detailed classification report.

In [ ]:
# Evaluate on the held-out test set
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")

# Generate predictions
y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# Classification report: precision, recall, F1 per class
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(10)]))

## Step 9: Confusion Matrix

A confusion matrix shows which digits are most often confused with each other.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

## Step 10: Visualize Sample Predictions

Display 25 random test images with true and predicted labels. Green = correct, Red = incorrect.

In [ ]:
indices = np.random.choice(len(X_test), 25, replace=False)
fig, axes = plt.subplots(5, 5, figsize=(14, 14))

for plot_idx, data_idx in enumerate(indices):
    ax = axes.flatten()[plot_idx]
    true_label = y_test[data_idx]
    pred_label = y_pred[data_idx]
    color = 'green' if true_label == pred_label else 'red'

    ax.imshow(X_test[data_idx].squeeze(), cmap='gray')
    ax.set_title(f'True: {true_label} | Pred: {pred_label}', color=color, fontsize=10)
    ax.axis('off')

plt.suptitle('Sample Predictions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'sample_predictions.png', dpi=150)
plt.show()

## Step 11: Save the Model

Save in both `.keras` (recommended) and `.h5` (backward compatible) formats.

In [ ]:
model.save(str(MODELS_DIR / 'cnn_model.keras'))
model.save(str(MODELS_DIR / 'cnn_model.h5'))
print(f"Model saved to {MODELS_DIR}")

## Step 12: Predict on Custom Images

Use OpenCV to load, preprocess, and predict digits from your own images. Place images in the `images/` folder.

In [ ]:
def predict_custom_image(image_path):
    """Load, preprocess, and predict a custom handwritten digit image."""
    # Read and convert to grayscale
    image = cv2.imread(str(image_path))
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (28, 28))

    # Invert if light background (MNIST uses white on black)
    if np.mean(resized) > 127:
        resized = 255 - resized

    # Normalize and reshape
    processed = resized.astype('float32') / 255.0
    processed = processed.reshape(1, 28, 28, 1)

    # Predict
    probs = model.predict(processed, verbose=0)[0]
    digit = int(np.argmax(probs))
    confidence = probs[digit] * 100

    print(f"Predicted Digit: {digit}")
    print(f"Confidence: {confidence:.1f}%")

    plt.figure(figsize=(4, 4))
    plt.imshow(processed.squeeze(), cmap='gray')
    plt.title(f'Predicted: {digit} ({confidence:.1f}%)')
    plt.axis('off')
    plt.show()

# Example: predict from images/ folder (uncomment when you have an image)
# predict_custom_image(PROJECT_ROOT / 'images' / 'my_digit.png')